In [0]:
%run ../common/config

In [0]:
encounters_df= spark.read.table(f"{env_catalog}.bronze.encounters")

In [0]:
encounters_df.printSchema()


In [0]:
from pyspark.sql.functions import *

In [0]:
silver_encounters = (
    encounters_df
    .dropDuplicates(["Id"])
    .withColumn(
        "encounter_start",
        to_timestamp(col("START"))
    )
    .withColumn(
        "encounter_end",
        to_timestamp(col("STOP"))
    )
    .select(
        col("Id").alias("encounter_id"),
        col("PATIENT").alias("patient_id"),
        col("ORGANIZATION").alias("organization_id"),
        col("PROVIDER").alias("provider_id"),
        col("PAYER").alias("payer_id"),
        "encounter_start",
        "encounter_end",
        col("ENCOUNTERCLASS").alias("encounter_class"),
        col("CODE").alias("encounter_code"),
        col("DESCRIPTION").alias("description"),
        col("BASE_ENCOUNTER_COST").alias("base_encounter_cost"),
        col("TOTAL_CLAIM_COST").alias("total_claim_cost"),
        col("PAYER_COVERAGE").alias("payer_coverage"),
        col("REASONCODE").alias("reason_code"),
        col("REASONDESCRIPTION").alias("reason_description"),
        "source_file",
        "source_system",
        "load_timestamp"
    )
    .withColumn(
        "silver_load_timestamp",
        current_timestamp()
    )
)

In [0]:
valid_encounters = silver_encounters.filter(
    col("encounter_id").isNotNull() &
    col("patient_id").isNotNull() &
    col("provider_id").isNotNull()
)

In [0]:
(
    valid_encounters.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        f"{env_catalog}.silver.encounters"
    )
)

In [0]:
%sql
SELECT COUNT(*)
FROM dev_healthcare.silver.encounters;